# Attendance Prediction Model

**Data**: synthetic, generated by `generate_dataset.py`. Each event's `actual_attendance` is `expected_audience` scaled by hand-picked multiplicative effects for event type, day, time, location, promotion channel, and budget, plus random noise (see that script's docstring for the exact effect sizes and why they exist).

**Important limitation**: because the effect sizes are hand-picked rather than fit to real event outcomes, this model demonstrates the ML pipeline (feature engineering, train/test split, model selection, evaluation) rather than a validated real-world predictor. Swapping in a real historical dataset with the same column schema would let this pipeline produce genuinely predictive results.

**Data leakage fix (v2)**: the original version of this notebook used `feedback_score` as an input feature. That's a logical error, not just a modeling nitpick: feedback is collected *after* an event happens, so an organizer predicting attendance beforehand can't possibly know it. It's now generated as a downstream *outcome* of attendance (via turnout ratio) and excluded from the model's inputs entirely.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [ ]:
df = pd.read_csv('../data/event_data.csv')
df.head()

## Encode categoricals

In [ ]:
le_event = LabelEncoder()
le_day = LabelEncoder()
le_time = LabelEncoder()
le_location = LabelEncoder()
le_promo = LabelEncoder()

df['event_type_enc'] = le_event.fit_transform(df['event_type'])
df['day_enc'] = le_day.fit_transform(df['day_of_week'])
df['time_enc'] = le_time.fit_transform(df['start_time'])
df['location_enc'] = le_location.fit_transform(df['location'])
df['promo_enc'] = le_promo.fit_transform(df['promotion_channel'])

df.head()

## Features

`feedback_score` is intentionally excluded -- see limitation note above. Everything here is knowable *before* the event happens, which is the only version of this problem that's actually useful to an organizer.

In [ ]:
FEATURES = [
    'event_type_enc', 'day_enc', 'time_enc', 'location_enc', 'promo_enc',
    'expected_audience', 'budget'
]

X = df[FEATURES]
y = df['actual_attendance']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Baselines

Before trusting the model's R², check it against two baselines:
1. **Naive**: just predict `actual_attendance = expected_audience`.
2. **Linear regression**: a simple model with no interaction effects.

If the Random Forest doesn't clearly beat both, the "ML" isn't adding value.

In [6]:
naive_pred = X_test['expected_audience']
print('Naive baseline R2:', r2_score(y_test, naive_pred))

lr = LinearRegression().fit(X_train, y_train)
print('Linear regression R2:', r2_score(y_test, lr.predict(X_test)))

Naive baseline R2: 0.5965218007296638
Linear regression R2: 0.7337450406514725


In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

In [8]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 47.4999
RMSE: 65.47944713992322
R2: 0.8299782766926215


In [ ]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

fig = px.bar(
    importance,
    x='Importance',
    y='Feature',
    orientation='h',
    color='Importance',
    title='Feature Importance for Attendance Prediction',
    template='plotly_dark'
)
fig.update_layout(title_x=0.5)
fig.show()

importance

## Save model + encoders

Encoders are saved alongside the model so the mapping from category -> integer doesn't have to be hardcoded and kept in sync by hand in the app layer.

In [10]:
joblib.dump(model, '../models/attendance_model.pkl')
joblib.dump(
    {'event': le_event, 'day': le_day, 'time': le_time,
     'location': le_location, 'promo': le_promo},
    '../models/label_encoders.pkl'
)
print("Model and encoders saved!")

Model and encoders saved!
